# 深度学习计算总结

本notebook整理了《动手学深度学习》第五章的核心概念和代码，涵盖深度学习框架的关键组件。

## 本章结构
1. **层和块（Layers and Blocks）** — 模型构建的基本单元
2. **参数管理（Parameter Management）** — 访问、初始化、共享参数
3. **延后初始化（Deferred Init）** — 框架自动推断参数形状
4. **自定义层（Custom Layers）** — 创建自己的层
5. **读写文件（Read/Write）** — 保存和加载模型
6. **GPU使用** — 利用GPU加速计算

## 一、层和块（Layers and Blocks）

### 1.1 核心概念

神经网络的基本组成单位是**块（Block）**：
- **层（Layer）**：单个层，如全连接层
- **块（Block）**：由多个层组成的组件，甚至可以是整个模型

**块的特点**：
1. 接受输入数据
2. 通过前向传播生成输出
3. 存储和访问参数
4. 自动计算梯度（反向传播）

### 1.2 nn.Sequential

最简单的块容器，按顺序连接各层：

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F

# 使用nn.Sequential构建模型
net = nn.Sequential(
    nn.Linear(20, 256),  # 全连接层：20维输入 → 256维输出
    nn.ReLU(),           # ReLU激活函数
    nn.Linear(256, 10)   # 全连接层：256维输入 → 10维输出
)

X = torch.rand(2, 20)  # 2个样本，每个20维
output = net(X)         # 前向传播
print(f'输入shape: {X.shape}')
print(f'输出shape: {output.shape}')

### 1.3 自定义块

通过继承`nn.Module`创建自定义块：

In [ ]:
class MLP(nn.Module):
    """自定义多层感知机块"""
    def __init__(self):
        super().__init__()  # 调用父类构造函数
        self.hidden = nn.Linear(20, 256)  # 隐藏层
        self.out = nn.Linear(256, 10)     # 输出层

    def forward(self, X):
        """前向传播：输入 → 隐藏层(ReLU) → 输出层"""
        return self.out(F.relu(self.hidden(X)))

# 使用自定义块
net = MLP()
output = net(X)
print(f'自定义MLP输出shape: {output.shape}')

### 1.4 自定义Sequential

理解`nn.Sequential`的工作原理：

In [ ]:
class MySequential(nn.Module):
    """自定义Sequential容器"""
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            # 将模块存储在_modules字典中（OrderedDict）
            self._modules[str(idx)] = module

    def forward(self, X):
        """按顺序执行每个模块"""
        for block in self._modules.values():
            X = block(X)
        return X

# 使用自定义Sequential
net = MySequential(
    nn.Linear(20, 256),
    nn.ReLU(),
    nn.Linear(256, 10)
)
output = net(X)
print(f'MySequential输出shape: {output.shape}')

### 1.5 在前向传播中执行任意代码

块不仅限于顺序结构，可以在前向传播中执行任意代码：

In [ ]:
class FixedHiddenMLP(nn.Module):
    """包含固定参数和控制流的块"""
    def __init__(self):
        super().__init__()
        # 不计算梯度的随机权重（常量参数）
        self.rand_weight = torch.rand((20, 20), requires_grad=False)
        self.linear = nn.Linear(20, 20)

    def forward(self, X):
        X = self.linear(X)                    # 线性变换
        X = F.relu(torch.mm(X, self.rand_weight) + 1)  # 使用常量参数
        X = self.linear(X)                    # 复用全连接层（共享参数）
        # 控制流：while循环
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()

net = FixedHiddenMLP()
output = net(X)
print(f'FixedHiddenMLP输出: {output}')

## 二、参数管理（Parameter Management）

### 2.1 参数访问

访问模型参数的几种方式：

In [ ]:
# 创建模型
net = nn.Sequential(
    nn.Linear(4, 8),   # 第0层
    nn.ReLU(),         # 第1层
    nn.Linear(8, 1)    # 第2层
)
X = torch.rand(size=(2, 4))
net(X)

# 方法1：通过索引访问某一层的参数
print('=== 方法1：索引访问 ===')
print(net[2].state_dict())  # 第2层（输出层）的参数

# 方法2：访问具体参数
print('\n=== 方法2：访问具体参数 ===')
print(f'参数类型: {type(net[2].bias)}')
print(f'参数值: {net[2].bias.data}')
print(f'梯度: {net[2].weight.grad}')  # 未调用反向传播前为None

In [ ]:
# 方法3：一次性访问所有参数
print('=== 方法3：所有参数 ===')
for name, param in net.named_parameters():
    print(f'{name}: shape={param.shape}')

# 方法4：通过state_dict访问
print('\n=== 方法4：state_dict ===')
print(net.state_dict()['2.bias'].data)

### 2.2 参数初始化

PyTorch提供多种内置初始化方法：

In [ ]:
# 方法1：正态分布初始化
def init_normal(m):
    if type(m) == nn.Linear:
        nn.init.normal_(m.weight, mean=0, std=0.01)  # 均值0，标准差0.01
        nn.init.zeros_(m.bias)                        # 偏置初始化为0

net.apply(init_normal)
print(f'正态初始化: {net[0].weight.data[0]}')

# 方法2：常数初始化
def init_constant(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 1)  # 全部初始化为1
        nn.init.zeros_(m.bias)

net.apply(init_constant)
print(f'常数初始化: {net[0].weight.data[0]}')

# 方法3：Xavier初始化（推荐）
def init_xavier(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)  # Xavier均匀分布

net[0].apply(init_xavier)
print(f'Xavier初始化: {net[0].weight.data[0]}')

### 2.3 参数绑定（共享参数）

多个层可以共享同一组参数：

In [ ]:
# 定义共享层
shared = nn.Linear(8, 8)

# 构建模型，第2层和第4层共享参数
net = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    shared,      # 第2层
    nn.ReLU(),
    shared,      # 第4层（与第2层共享参数）
    nn.ReLU(),
    nn.Linear(8, 1)
)

X = torch.rand(size=(2, 4))
net(X)

# 验证参数是否相同
print('第2层和第4层权重相同:', (net[2].weight.data[0] == net[4].weight.data[0]).all().item())

# 修改一个，另一个也会变
net[2].weight.data[0, 0] = 100
print('修改后仍然相同:', (net[2].weight.data[0] == net[4].weight.data[0]).all().item())

## 三、延后初始化（Deferred Initialization）

### 3.1 核心概念

深度学习框架的**延后初始化**机制：
- 定义网络时不需要指定输入维度
- 框架在第一次前向传播时自动推断每层的大小
- 然后自动初始化参数

**好处**：
- 简化模型定义
- 避免手动计算维度
- 方便修改模型架构

In [ ]:
# 定义网络时不需要指定输入维度
net = nn.Sequential(
    nn.Linear(20, 256),  # 这里20是输入维度，需要指定
    nn.ReLU(),
    nn.Linear(256, 10)   # 256由上一层决定
)

# 在第一次前向传播之前，参数尚未初始化
print('第一次前向传播前，参数shape未知')

# 第一次前向传播时，框架自动初始化参数
X = torch.rand(2, 20)
net(X)

# 现在参数已经初始化
print('第一次前向传播后，参数已初始化')
for name, param in net.named_parameters():
    print(f'{name}: {param.shape}')

## 四、自定义层（Custom Layers）

### 4.1 不带参数的层

In [ ]:
class CenteredLayer(nn.Module):
    """自定义层：减去均值（不带参数）"""
    def __init__(self):
        super().__init__()

    def forward(self, X):
        return X - X.mean()  # 减去均值

# 测试
layer = CenteredLayer()
print(f'输入: [1, 2, 3, 4, 5]')
print(f'输出: {layer(torch.FloatTensor([1, 2, 3, 4, 5]))}')

# 作为组件嵌入到更大模型中
net = nn.Sequential(nn.Linear(8, 128), CenteredLayer())
Y = net(torch.rand(4, 8))
print(f'输出均值（应接近0）: {Y.mean().item():.10f}')

### 4.2 带参数的层

In [ ]:
class MyLinear(nn.Module):
    """自定义全连接层（带参数）"""
    def __init__(self, in_units, units):
        super().__init__()
        # 使用nn.Parameter创建可训练参数
        self.weight = nn.Parameter(torch.randn(in_units, units))
        self.bias = nn.Parameter(torch.randn(units,))

    def forward(self, X):
        linear = torch.matmul(X, self.weight.data) + self.bias.data
        return F.relu(linear)  # 线性变换 + ReLU

# 测试自定义层
linear = MyLinear(5, 3)
print(f'权重shape: {linear.weight.shape}')
print(f'偏置shape: {linear.bias.shape}')

# 用自定义层构建模型
net = nn.Sequential(MyLinear(64, 8), MyLinear(8, 1))
output = net(torch.rand(2, 64))
print(f'模型输出shape: {output.shape}')

## 五、读写文件（Read/Write）

### 5.1 保存和加载张量

In [ ]:
# 保存单个张量
x = torch.arange(4)
torch.save(x, 'x-file')  # 保存到文件

# 加载张量
x2 = torch.load('x-file')
print(f'加载的张量: {x2}')

# 保存张量列表
y = torch.zeros(4)
torch.save([x, y], 'x-files')
x2, y2 = torch.load('x-files')
print(f'加载的列表: {x2}, {y2}')

# 保存字典
mydict = {'x': x, 'y': y}
torch.save(mydict, 'mydict')
mydict2 = torch.load('mydict')
print(f'加载的字典: {mydict2}')

### 5.2 保存和加载模型参数

**重要**：保存的是参数（state_dict），不是整个模型。
加载时需要先创建模型架构，再加载参数。

In [ ]:
class MLP(nn.Module):
    """多层感知机"""
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.output = nn.Linear(256, 10)

    def forward(self, x):
        return self.output(F.relu(self.hidden(x)))

# 创建并训练模型
net = MLP()
X = torch.randn(size=(2, 20))
Y = net(X)  # 原始模型的输出

# 保存模型参数
torch.save(net.state_dict(), 'mlp.params')
print('模型参数已保存到 mlp.params')

# 加载模型参数
clone = MLP()  # 先创建相同架构的模型
clone.load_state_dict(torch.load('mlp.params'))  # 再加载参数
clone.eval()  # 设置为评估模式

# 验证：相同输入产生相同输出
Y_clone = clone(X)
print(f'输出相同: {(Y_clone == Y).all().item()}')

## 六、GPU使用

### 6.1 计算设备

PyTorch中指定计算设备：
- `torch.device('cpu')`：CPU
- `torch.device('cuda')` 或 `torch.device('cuda:0')`：第一个GPU
- `torch.device('cuda:1')`：第二个GPU

In [ ]:
import torch
from torch import nn

# 查询可用GPU数量
print(f'可用GPU数量: {torch.cuda.device_count()}')

# 便捷函数
def try_gpu(i=0):
    """如果存在，则返回gpu(i)，否则返回cpu()"""
    if torch.cuda.device_count() >= i + 1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')

def try_all_gpus():
    """返回所有可用的GPU，如果没有GPU则返回[cpu()]"""
    devices = [torch.device(f'cuda:{i}')
               for i in range(torch.cuda.device_count())]
    return devices if devices else [torch.device('cpu')]

print(f'GPU 0: {try_gpu()}')
print(f'GPU 10: {try_gpu(10)}')  # 不存在时返回cpu
print(f'所有GPU: {try_all_gpus()}')

### 6.2 张量与GPU

In [ ]:
# 默认在CPU上创建
x = torch.tensor([1, 2, 3])
print(f'默认设备: {x.device}')

# 在GPU上创建张量
X = torch.ones(2, 3, device=try_gpu())
print(f'GPU上的张量: {X}')

# 复制到另一个设备
if torch.cuda.device_count() >= 2:
    Y = torch.rand(2, 3, device=try_gpu(1))  # 在第二个GPU上
    Z = X.cuda(1)  # 将X复制到第二个GPU
    print(f'X在: {X.device}')
    print(f'Z在: {Z.device}')
    print(f'Y+Z: {Y + Z}')  # 在同一设备上计算
else:
    print('只有一个GPU，跳过多GPU演示')

### 6.3 神经网络与GPU

In [ ]:
# 将模型放到GPU上
net = nn.Sequential(nn.Linear(3, 1))
net = net.to(device=try_gpu())  # 移动到GPU

# 在GPU上计算
X_gpu = torch.ones(2, 3, device=try_gpu())
output = net(X_gpu)
print(f'输出设备: {output.device}')
print(f'参数设备: {net[0].weight.data.device}')

## 七、本章核心知识总结

### 核心概念对比

| 概念 | 英文 | 说明 |
|------|------|------|
| 层 | Layer | 单个网络层，如nn.Linear |
| 块 | Block | 由多个层组成的组件 |
| Module | Module | PyTorch中表示块的基类 |
| Sequential | Sequential | 按顺序连接各层的容器 |
| state_dict | state_dict | 模型参数的字典 |
| 延后初始化 | Deferred Init | 第一次前向传播时才初始化参数 |
| 参数绑定 | Parameter Sharing | 多个层共享同一组参数 |

### 关键API速查表

| 功能 | API | 说明 |
|------|-----|------|
| 创建全连接层 | `nn.Linear(in, out)` | 输入in维，输出out维 |
| 顺序容器 | `nn.Sequential(...)` | 按顺序连接层 |
| 自定义块 | 继承`nn.Module` | 实现`__init__`和`forward` |
| 访问参数 | `net.parameters()` | 获取所有参数 |
| 参数字典 | `net.state_dict()` | 获取参数字典 |
| 初始化权重 | `nn.init.xavier_uniform_()` | Xavier初始化 |
| 保存参数 | `torch.save(net.state_dict(), path)` | 保存到文件 |
| 加载参数 | `net.load_state_dict(torch.load(path))` | 从文件加载 |
| 指定设备 | `tensor.to(device)` | 移动到指定设备 |
| GPU设备 | `torch.device('cuda:0')` | 第一个GPU |

### 常见操作

```python
# 1. 定义模型
net = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 10))

# 2. 访问某一层参数
net[0].weight.data  # 第0层的权重
net[0].bias.data    # 第0层的偏置

# 3. 初始化参数
nn.init.normal_(net[0].weight, std=0.01)
nn.init.zeros_(net[0].bias)

# 4. 保存和加载
torch.save(net.state_dict(), 'model.params')
net.load_state_dict(torch.load('model.params'))

# 5. 使用GPU
net = net.to('cuda:0')  # 模型放到GPU
X = X.to('cuda:0')      # 数据放到GPU
```